<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Ensemble Methods and Unsupervised Learning — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked answers to the **» Predict**, **» Investigate**, **» Modify** and **» Make** tasks in **07_Ensemble_Methods_and_Unsupervised_Learning**, in the same order. Run the setup cell first; it rebuilds the table and the split as the session did. The numbers quoted are for the course views and `random_state=123`. Where a task asked for a decision, the answer here is one defensible choice, not the only one.*

## Setup

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq
import sklearn

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

# The model's columns: Session 4's, without diabetes, which is only coded after discharge
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
features = numeric_features + ['temperature_mean'] + kept_binary

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))
from sklearn.model_selection import train_test_split

patients = df_features['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

in_train = df_features['subject_id'].isin(train_patients)
in_test = df_features['subject_id'].isin(test_patients)
df_train = df_features[in_train].copy()
df_test = df_features[in_test].copy()

y_train = df_train['died']
y_test = df_test['died']
print(len(df_train), 'training rows,', len(df_test), 'test rows')
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.fit(df_train[features])

X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])

## From one tree to a forest

### Investigate: one tree, grown until it stops
**Model C.** The tree is 29 levels deep, with **709 leaves for 6,738 training rows**, and **306** of those leaves hold a single patient. It scores **1.000** on the training rows and **0.604** on the test rows, the worst score of any model in the course. With no limit on its size it keeps splitting until each leaf holds patients with one outcome: it has learned which training patients died, not why. The groups' tree in Session 6 scored 0.756 because its search chose `max_depth = 8` and leaves of at least 100 patients, which hold it back.

### Predict: ten trees from bootstrap samples
**Alike from the top, different underneath.** Seven of the ten split first on `resp_rate_mean`, and most of those next on `temperature_mean` and `sbp_min`; two start with `temperature_mean`, one with `sbp_min`. Each has 460 to 510 leaves, and scores 0.58 to 0.63 on the test patients.

### Who do the trees flag?
Each tree flags 179 to 244 test patients, but together they flag **877** different patients: 372 by only one tree, 427 by two to five, and 78 by six or more. Each tree catches about **61** of the 197 deaths in the test rows; **159** are caught by at least one tree.

### Investigate: what could you do with ten trees like these?
**Average them.** Each tree's mistakes are its own: the small leaves differ from sample to sample, and so do the patients they flag. What the trees share is the pattern that every sample contains. Their disagreement is **variance**, and averaging their risks cancels much of it while keeping what they share: the average of just these ten trees scores **0.739** on the test patients, against about 0.60 for each. That is bagging, and with 300 trees it is the next Predict.

### Predict: a forest of 300 trees
**0.796**, from trees that each score about 0.60: close to logistic regression's 0.801, and far above the groups' tuned single tree. Averaging did what no setting of one tree could. A deep tree's risks are all exactly 0 or 1 (its AUC, 0.604, is that of a single yes/no cut-off); averaging 300 gives hundreds of levels, so the forest can rank patients.

### Investigate: how many trees?
| Trees | Training AUC | Test AUC |
| --: | --: | --: |
| 1 | 0.841 | 0.610 |
| 10 | 1.000 | 0.749 |
| 50 | 1.000 | 0.786 |
| 300 | 1.000 | 0.796 |

The gain slows after about 50 trees and stops by a few hundred: more trees never make a forest worse, only slower.

**Overfitting? A little, but you cannot see it here.** Every tree is fully grown and has memorised its sample, and each training patient is in the samples of about two thirds of the trees, so the forest scores 1.00 on the training rows whatever is happening on new patients. Averaging cancels most of what each tree memorised, not all: by cross-validation, leaves of at least 5 patients score 0.807 against 0.800 for fully grown trees (Part 1), a gap within the spread between folds. Judge a forest by cross-validation, or by its **out-of-bag** score, from the trees whose sample left each patient out (`oob_score=True`): 0.800 for this forest. More trees never make a forest worse; deeper trees can.

The random subsets of features matter too. The same 300 trees grown by bagging alone, with every feature available at each split (`max_features=None`):

| 300 trees | Different first splits | Correlation between trees' predictions | Test AUC |
| :-- | --: | --: | --: |
| Bagging only | 3 | 0.21 | 0.782 |
| Random forest | 18 | 0.17 | 0.796 |

Trees that differ more average better, though the gain here, 0.014, is about the size of the variation between random seeds.

## Boosting

### Investigate: the 38 deaths no tree flagged
**No.** Averaging cancels the mistakes the trees make differently; a mistake every tree makes survives any average. To fix it, a tree must know what the others missed: grow trees one after another, each fitted to the training patients the trees so far get most wrong. That is boosting.

### Investigate: the learning rate
| `learning_rate` | Training AUC | Test AUC |
| --: | --: | --: |
| 0.01 | 0.883 | 0.794 |
| 0.03 | 0.936 | **0.806** |
| 0.1 (default) | 0.991 | 0.796 |
| 0.3 | 1.000 | 0.784 |
| 1 | 0.694 | 0.649 |

**0.03.** At 0.01, 100 trees of small steps have not gone far enough: underfitting, though more trees would fix it. From 0.1 upwards the training AUC climbs to 1.000 while the test AUC falls: each tree corrects so much that the model fits the training patients' noise. At 1, each correction overshoots, and the model is worse even on the training rows. The best rate depends on the number of trees, which is why Part 2 tunes them together.

The figure in the appendix, "What the learning rate does", shows why. Every model starts each patient at the training death rate, 0.111, and each tree moves the risks a fraction of the way its leaves suggest. At **0.01** the steps are so small that after 100 trees the deaths' average risk has only reached 0.21, and is still rising. At **0.1** the two groups separate steadily, the deaths to 0.29 and the survivors to 0.08, and level off by about 50 trees. At **1** the first tree moves the deaths straight to 0.32, and every later tree overcorrects the one before, so the risks jump about and the test AUC falls.

## Class imbalance

### Predict: oversampling the deaths
The test AUC **does not change**: 0.801 as it is, 0.800 oversampled. The average predicted risk of death rises from **0.112 to 0.395**, against 0.118 who actually died. Oversampling told the model that deaths are as common as survivals, so it raised every patient's risk, but kept them in nearly the same order, and the AUC measures only the order.

### Resampling moves the threshold
At 0.5, the oversampled model flags **32.7%** of the test patients and finds **71%** of those who died. The unchanged model, with its threshold lowered to 0.111, the share who died in training, flags **32.4%** and also finds **71%**. Oversampling bought nothing a threshold could not, and cost the meaning of the risks.

### Predict: oversampling before cross-validation
**1.000.** The copies of each patient who died are spread across the folds by the shuffle, so most validation deaths are exact copies of training rows, and a forest of deep trees recognises them. It looks like a perfect model and is the same forest as before.

### Investigate: where did the difference go?
With the oversampler inside the pipeline and folds by patient, **0.786**: no better than the same forest without resampling, 0.797. The difference was leakage: the same patients in training and validation. Session 5's rule, **folds by patient**, would have caught most of it on its own: oversampled first but split with `GroupKFold`, the copies share their `subject_id` and stay in one fold, and the forest scores 0.798.

## Unsupervised learning

### Investigate: what has gone wrong?
The clusters' mean heart rates are **69, 87 and 108** beats per minute; their mean respiratory rates only **17.9, 19.0 and 21.8** breaths per minute. k-means used heart rate and almost ignored respiratory rate. It measures distance in the columns' own units, and heart rate varies over a range several times wider, so it dominates every distance. Session 4 scaled features for the same reason.

### Investigate: scale first
Now both count. Three groups a clinician might name: a **calmer** group, heart rate 72 and respiratory rate 17.5, 4,079 patients of whom **6%** died; a **tachycardic** group, heart rate 95 with a normal respiratory rate, **11%**; and a **tachypnoeic** group, respiratory rate 25 and heart rate 98, **24%**. k-means never saw `died`, yet the clusters differ fourfold in risk. Before calling one a phenotype you would want it to appear again with a different *k*, a different sample and at another hospital, and to mean something clinically beyond "sicker".

### Investigate: three heart-rate columns
**About one.** The first component holds **85%** of the variation, with nearly equal loadings, 0.56, 0.62 and 0.55: it measures the overall level of the heart rate. The second, **13%**, has loadings of −0.68 on the minimum and 0.73 on the maximum: how much the heart rate varied during the day. The third holds 2%. Three columns, two numbers' worth of information.

## Independent practice

### Part 1, Modify: settings for a forest
| `min_samples_leaf` | `max_features` | Mean AUC | Spread between folds |
| --: | :-- | --: | --: |
| 1 | `'sqrt'` | 0.800 | 0.023 |
| 5 | `'sqrt'` | **0.807** | 0.023 |
| 20 | `'sqrt'` | 0.805 | 0.019 |
| 100 | `'sqrt'` | 0.794 | 0.017 |
| 5 | 0.5 | 0.799 | 0.022 |

Leaves of at least 5 patients do best, but every setting is within the spread between folds of the others: a forest is not sensitive to its settings, which is one reason it is a good first model. Leaves of 100 make each tree too simple, and the bias is back. Half the features at each split makes the trees more alike, so their average gains less. The cell below runs all five, about 30 seconds.

In [ ]:
# Leaf 5 does best, 0.807, but all within the spread between folds, 0.02. max_features=0.5 is worse
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

settings = [(1, 'sqrt'), (5, 'sqrt'), (20, 'sqrt'), (100, 'sqrt'), (5, 0.5)]
for leaf, share in settings:
    forest_settings = RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, max_features=share, random_state=0)
    pipeline_forest = make_pipeline(SimpleImputer(strategy='median'), forest_settings)
    scores = cross_val_score(
        pipeline_forest,
        df_train[features],
        df_train['died'],
        cv=GroupKFold(n_splits=5),
        groups=df_train['subject_id'],
        scoring='roc_auc',
    )
    print(leaf, share, ' mean AUC', round(scores.mean(), 3), ' spread', round(scores.std(), 3))

### Part 2, Make: tune boosting, then test it once
The grid is `{'histgradientboostingclassifier__learning_rate': [0.01, 0.03, 0.1], 'histgradientboostingclassifier__max_iter': [100, 300]}`, and the blanks are `df_train['subject_id']` and `df_test[features]`. The search chooses **`learning_rate = 0.03` with 100 trees**: cross-validated AUC **0.806 ± 0.021**. Tested once, **0.806**.

100 trees is the edge of the grid, so extend it with 50:

| `learning_rate` | 50 trees | 100 trees | 300 trees |
| --: | --: | --: | --: |
| 0.01 | 0.783 | 0.795 | 0.805 |
| 0.03 | 0.801 | **0.806** | 0.795 |
| 0.1 | 0.801 | 0.793 | 0.775 |

The choice stands. The table shows the trade-off: 0.01 with 300 trees scores the same as 0.03 with 100, and 0.1 with 50 the same as 0.03 with 50. Smaller steps need more trees; past the best number, more trees overfit, as the loss curve did for Session 6's network.

In [ ]:
# Chosen: learning_rate 0.03, 100 trees. CV AUC 0.806, test AUC 0.806. 100 is the edge: 50 does worse, so it stands
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline

pipeline_boost = make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(random_state=0))

# 1. The grid: in a pipeline, each setting's name starts with its step's
grid = {
    'histgradientboostingclassifier__learning_rate': [0.01, 0.03, 0.1],
    'histgradientboostingclassifier__max_iter': [100, 300],
}

# 2. The search, by cross-validation on the training rows, one patient per fold
search = GridSearchCV(pipeline_boost, grid, cv=GroupKFold(n_splits=5), scoring='roc_auc')
search.fit(df_train[features], df_train['died'], groups=df_train['subject_id'])
print('Chosen:', search.best_params_, ' CV AUC:', round(search.best_score_, 3))

# 3. Once, at the end: the chosen model on the test rows
prob_search = search.predict_proba(df_test[features])[:, 1]
print('Test AUC:', round(roc_auc_score(df_test['died'], prob_search), 3))

### Part 3, Investigate: three ways to bend
With the settings chosen in Parts 1 and 2: leaves of at least 5 patients for the forest, and boosting at a learning rate of 0.03 with 100 trees.

| Model | Folds | Mean |
| :-- | :-- | --: |
| Logistic regression | 0.802, 0.795, 0.798, 0.788, 0.753 | 0.787 |
| Logistic regression + distances | 0.819, 0.817, 0.815, 0.792, 0.770 | 0.803 |
| Network, (50,), `alpha` = 1 | 0.820, 0.822, 0.818, 0.802, 0.775 | 0.807 |
| Forest, leaf 5 | 0.835, 0.817, 0.825, 0.776, 0.783 | 0.807 |
| Boosting, rate 0.03 | 0.822, 0.813, 0.830, 0.785, 0.777 | 0.806 |

**All four beat logistic regression in every fold.** None is ahead of the others by more than the spread between folds: their means are within 0.004, and which one wins changes from fold to fold. The forest wins two folds, the network two, boosting one. Four very different models, two of them given no help with the shapes, arrive at the same place.

### Part 3, Modify: give the trees the distances
**No gain:** the forest falls slightly, from 0.807 to 0.798, and boosting from 0.806 to 0.803. A tree can split temperature at 36 °C and again at 38 °C, so it finds "far from normal" by itself; the distances only repeat what it can already see, and give the forest's random subsets more columns that say the same thing. Logistic regression gained because a straight line cannot do that. Feature engineering helps the model that cannot find the shape.

Part 3's setup, as in the session: Session 6's `fold_aucs`, and the training table with the five distances.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def fold_aucs(model, table, columns):
    """The AUC on each of the same five folds of the training patients."""
    pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), model)
    return cross_val_score(pipeline, table[columns], table['died'], cv=GroupKFold(n_splits=5),
                           groups=table['subject_id'], scoring='roc_auc')


table = df_train.copy()
table['temperature_from_normal'] = (table['temperature_mean'] - 37).abs()
table['heart_rate_mean_from_normal'] = (table['heart_rate_mean'] - 80).abs()
table['sbp_min_from_normal'] = (table['sbp_min'] - 110).abs()
table['resp_rate_mean_from_normal'] = (table['resp_rate_mean'] - 16).abs()
table['glucose_max_from_normal'] = (table['glucose_max'] - 120).abs()
distances = ['temperature_from_normal', 'heart_rate_mean_from_normal', 'sbp_min_from_normal',
             'resp_rate_mean_from_normal', 'glucose_max_from_normal']
features_plus = features + distances

In [ ]:
# Forest 0.807 -> 0.798, boosting 0.806 -> 0.803: a tree finds "far from normal" by splitting twice
aucs_forest_plus = fold_aucs(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0), table, features_plus)
aucs_boost_plus = fold_aucs(HistGradientBoostingClassifier(learning_rate=0.03, random_state=0), table, features_plus)

print('Forest + distances:  ', aucs_forest_plus.round(3), ' mean', round(aucs_forest_plus.mean(), 3))
print('Boosting + distances:', aucs_boost_plus.round(3), ' mean', round(aucs_boost_plus.mean(), 3))

### Part 4, Make: does oversampling help boosting?
| Boosting, rate 0.03, 100 trees | Folds | Mean |
| :-- | :-- | --: |
| As it is | 0.822, 0.813, 0.830, 0.785, 0.777 | **0.806** |
| Oversampled inside the pipeline | 0.824, 0.809, 0.821, 0.782, 0.770 | 0.801 |

**No: it is slightly worse in four of the five folds**, and ahead only in the first, by 0.002. Boosting already fits each new tree to the errors, which are largest for the deaths it misses, so copying the deaths adds little. Weighting them instead, `class_weight='balanced'`, gives 0.798. As for logistic regression in section 4, the imbalance here is not a problem to fix: 748 deaths in training are enough to learn from.

In [ ]:
# No: slightly worse in four of the five folds, 0.801 against 0.806
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

# Your settings from Part 2, as in Part 3
boosting_tuned = HistGradientBoostingClassifier(learning_rate=0.03, max_iter=100, random_state=0)

pipeline_boost_plain = make_pipeline(SimpleImputer(strategy='median'), boosting_tuned)
aucs_boost_plain = cross_val_score(
    pipeline_boost_plain,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('As it is:   ', aucs_boost_plain.round(3), ' mean', round(aucs_boost_plain.mean(), 3))

# The same model, with the oversampler inside the pipeline
pipeline_boost_over = make_imb_pipeline(SimpleImputer(strategy='median'), RandomOverSampler(random_state=1), boosting_tuned)
aucs_boost_over = cross_val_score(
    pipeline_boost_over,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Oversampled:', aucs_boost_over.round(3), ' mean', round(aucs_boost_over.mean(), 3))

### Modify (stretch): weights or a threshold?
| Version | Test AUC | Average predicted risk | Flagged | Sensitivity |
| :-- | --: | --: | --: | --: |
| As it is, threshold 0.5 | 0.801 | 0.112 | 1.9% | 0.12 |
| `class_weight='balanced'`, 0.5 | 0.802 | 0.396 | 32.7% | 0.72 |
| As it is, threshold 0.111 | 0.801 | 0.112 | 32.4% | 0.71 |

**Changed:** the risks and who is flagged. **Not changed:** the AUC. At 0.5 the unweighted model is nearly useless as an alarm, finding 12% of the deaths, but that is the threshold's fault, not the model's. Weighting and lowering the threshold flag the same patients, and find the same 71 to 72% of the deaths. **Hand a clinician the unweighted model with a chosen threshold:** its 11% means 11%, whereas the weighted model tells the average patient 40% when 12% die. The threshold itself should come from the cost of a missed death against a false alarm, not from the share who died, which is only a starting point.

In [ ]:
# Changed: the risks and who is flagged. Not changed: the AUC. Hand over the unweighted model with a chosen threshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, roc_auc_score

versions = [('As it is, 0.5', None, 0.5), ('Balanced, 0.5', 'balanced', 0.5), ('As it is, 0.111', None, y_train.mean())]
for name, weight, threshold in versions:
    model_weighted = LogisticRegression(class_weight=weight)
    model_weighted.fit(X_train, y_train)
    prob_weighted = model_weighted.predict_proba(X_test)[:, 1]
    flagged = prob_weighted >= threshold
    print(name, ' AUC', round(roc_auc_score(y_test, prob_weighted), 3), ' average predicted risk', round(prob_weighted.mean(), 3),
          ' flagged', round(flagged.mean(), 3), ' sensitivity', round(recall_score(y_test, flagged), 2))

### Modify (stretch): clusters of patients
| *k* | Patients per cluster | Share who died |
| --: | :-- | :-- |
| 2 | 2,560; 4,178 | 17%; 7% |
| 3 | 1,738; 3,118; 1,882 | 23%; 9%; 4% |
| 4 | 1,322; 1,922; 1,575; 1,919 | 23%; 14%; 5%; 5% |
| 5 | 1,139; 1,965; 1,363; 1,831; 440 | 23%; 13%; 4%; 5%; 18% |
| 6 | 1,472; 1,801; 1,232; 1,049; 750; 434 | 18%; 7%; 4%; 6%; 23%; 17% |

**Yes, at every *k*.** One cluster with about 23% mortality appears at every *k* from 3 to 6, and so does one with about 4%: those are the groups to trust, because they do not depend on the choice of *k*. At *k* = 3 they are a group with a fast heart rate, a group of older patients with a slower one, and a younger group. The clusters in between split and merge as *k* changes, so they are as likely to be the algorithm's choices as the patients'. Nothing in the data says which *k* is right; the silhouette score, in the appendix's reading, is one guide.

In [ ]:
# Every k separates risk: a ~23% cluster and a ~4% cluster appear at every k from 3 to 6
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numeric = numeric_features + ['temperature_mean']
prepare_numeric = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
scaled_numeric = prepare_numeric.fit_transform(df_train[numeric])

for k in [2, 3, 4, 5, 6]:
    kmeans = KMeans(n_clusters=k, random_state=1)
    cluster = kmeans.fit_predict(scaled_numeric)
    df_clusters = df_train.assign(cluster=cluster)
    by_cluster = df_clusters.groupby('cluster')
    summary = by_cluster.agg(patients=('died', 'size'), died=('died', 'mean'))
    print('k =', k)
    print(summary.round(3))

### Make (stretch): how many components?
**Five.** The shares are 0.317, 0.177, 0.143, 0.130, 0.114, 0.098 and 0.022, so four components hold 77% and five hold **88%**. Only the last component is nearly empty: its loadings are 0.72 on `heart_rate_mean` and −0.69 on `heart_rate_max`, the difference between two columns that move together. The other columns each carry information of their own, so the seven columns cannot be squeezed much without losing some.

In [ ]:
# Five components for 80% (four hold 77%): apart from the two heart-rate columns, the seven overlap little
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(scaled_numeric)

cumulative = pca.explained_variance_ratio_.cumsum()
print('Share of variance, each component:', pca.explained_variance_ratio_.round(3))
print('Cumulative:                       ', cumulative.round(3))

## Closing

### The two questions
1. **What limits them: the table.** Four models built on different ideas reach the same 0.81, and none beats the others by more than the spread between folds. When very different models agree, the limit is the information in the 25 columns, Session 6's lesson about temperature again. To beat 0.81, change the table: better features, measured closer to the outcome or carrying something the first day's vital signs do not, and enough patients to show a real difference.
2. **Which first:** logistic regression, as the baseline every other model must beat, and it is the easiest to explain. Then a random forest or gradient boosting, which find shapes and interactions themselves and need little tuning. Move on when a model is clearly beaten across folds, not on one test split; and prefer the simpler one when the difference is within the spread.